# ReadMate OCR 서비스 테스트

이미지와 PDF 파일의 텍스트 추출 서비스를 단계별로 검증합니다.

## 셀 1: 환경 설정

In [ ]:
import sys
from pathlib import Path

project_root = Path('../../').resolve()
sys.path.insert(0, str(project_root))
print(f'프로젝트 루트: {project_root}')

from src.lib.utils.device import available_device
device = available_device()
print(f'디바이스: {device}')

## 셀 2: PaddleOCREngine 싱글톤 확인

In [ ]:
from src.services.ocr_service import PaddleOCREngine

engine1 = PaddleOCREngine()
engine2 = PaddleOCREngine()
assert engine1 is engine2
print('✓ PaddleOCREngine 싱글톤 확인됨')

## 셀 3: 이미지 OCR 테스트

**준비:** sample_image.jpg 파일을 현재 디렉토리에 배치하거나 경로를 수정하세요

In [ ]:
image_path = 'sample_image.jpg'
try:
    with open(image_path, 'rb') as f:
        img_bytes = f.read()
    result = engine1.recognize(img_bytes)
    print(f'✓ OCR 완료:')
    print(f'  엔진: {result.engine}')
    print(f'  박스: {len(result.boxes)}')
    print(f'  confidence: {result.avg_confidence:.3f}')
    print(f'  길이: {len(result.raw_text)} 글자')
except FileNotFoundError:
    print(f'⚠ {image_path} 미발견')
except Exception as e:
    print(f'✗ 오류: {e}')

## 셀 4: 텍스트형 PDF 테스트

In [ ]:
from src.services.pdf_service import PyPDFEngine

pdf_engine = PyPDFEngine(ocr_fallback=engine1)
pdf_path = 'sample_text.pdf'
try:
    with open(pdf_path, 'rb') as f:
        pdf_bytes = f.read()
    result = pdf_engine.extract(pdf_bytes)
    print(f'✓ PDF 추출:')
    print(f'  페이지: {result.page_count}')
    print(f'  스캔형: {result.is_scanned}')
    print(f'  길이: {len(result.text)} 글자')
except FileNotFoundError:
    print(f'⚠ {pdf_path} 미발견')
except Exception as e:
    print(f'✗ 오류: {e}')

## 셀 5: 스캔형 PDF → OCR 폴백 테스트

In [ ]:
scanned_path = 'sample_scanned.pdf'
try:
    with open(scanned_path, 'rb') as f:
        scanned_bytes = f.read()
    result = pdf_engine.extract(scanned_bytes)
    print(f'✓ 스캔형 PDF 추출:')
    print(f'  페이지: {result.page_count}')
    print(f'  스캔형: {result.is_scanned}')
    print(f'  길이: {len(result.text)} 글자')
except FileNotFoundError:
    print(f'⚠ {scanned_path} 미발견')
except Exception as e:
    print(f'✗ 오류: {e}')